# Figure GS02 (supplementary): calibration of the two length-conditioned entropy references (genome_entropy #101)

**Status:** draft, Gadi. The ID is provisional.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figGS02_z_hist.tsv.gz`, `data/figGS02_deciles.tsv.gz`, `data/figGS02_summary.tsv.gz` | Gadi `entropy_null/calibration/scores_{bac,arc}_{protein,three_di}.tsv.gz` | `37_calibration_scores.pbs`, which scores `32_entropy_percentile.py` against both references |

- **Source:** `linsalrob/genome_entropy` `main` at `97a792f`.
- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --only G02`. It produces counts in z bins 0.1 wide from −15 to 8, decile counts of the percentile, and the mean and SD of z. The summary reproduces the published `calibration_summary.tsv` to four decimal places.
- **Data:** the matched ORFs of the systematic figure sample (bacteria every 300th row, n = 1,028,059; archaea every 30th row, n = 315,638).
- **Definitions:**
  - The **empirical** reference is the distribution of real matched ORFs of the same length.
  - The **i.i.d.** reference is multinomial draws at the pooled composition.
  - **Off-axis rows:** i.i.d. z values outside [−15, 8] are not drawn; their number is printed below.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}

import math
from matplotlib.colors import LogNorm

CLASS = {"matched": "#0072B2", "unmatched": "#E69F00"}
CEILINGS = [(0.0, "log$_2$1"), (1.0, "log$_2$2"), (math.log2(3), "log$_2$3"), (2.0, "log$_2$4")]


def ceilings(ax, axis="y", label=False):
    for v, t in CEILINGS:
        (ax.axhline if axis == "y" else ax.axvline)(
            v, color=MUTED, lw=0.6, ls=(0, (1, 2)), zorder=3)
        if label:
            ax.text(v, 1.0, t, transform=ax.get_xaxis_transform(), rotation=90,
                    ha="right", va="top", fontsize=5.8, color=MUTED)


In [ ]:
zh = pd.read_csv(DATA / "figGS02_z_hist.tsv.gz", sep="\t")
dec = pd.read_csv(DATA / "figGS02_deciles.tsv.gz", sep="\t")
sm = pd.read_csv(DATA / "figGS02_summary.tsv.gz", sep="\t")
REF = {"empirical": "#0072B2", "iid": "#D55E00"}
grid = np.linspace(-6, 6, 400)
fig, axes = plt.subplots(2, 3, figsize=(7.2, 4.4), gridspec_kw={"width_ratios": [1, 1, 1.1]})
for i, dom in enumerate(("bacteria", "archaea")):
    for j, (a, lab) in enumerate((("protein", "protein"), ("three_di", "3Di"))):
        ax = axes[i, j]
        for ref, rl in (("iid", "i.i.d."), ("empirical", "empirical")):
            g = zh[(zh.domain == dom) & (zh.alphabet == a) & (zh.reference == ref)]
            s = sm[(sm.domain == dom) & (sm.alphabet == a) & (sm.reference == ref)].iloc[0]
            dens = g.n / (s.n * (g.z_hi - g.z_lo))
            ax.stairs(dens, np.append(g.z_lo, g.z_hi.iloc[-1]), fill=True, color=REF[ref],
                      alpha=0.5, label=f"{rl}: mean {s.z_mean:+.2f}, sd {s.z_sd:.2f}")
        ax.plot(grid, np.exp(-grid ** 2 / 2) / np.sqrt(2 * np.pi), color=INK, lw=0.7,
                ls="--", label="N(0, 1)")
        ax.set_xlim(-15, 8)
        ax.legend(frameon=False, fontsize=5.6, loc="upper left")
        if i == 0:
            ax.set_title(f"{lab} z")
        else:
            ax.set_xlabel("z")
        if j == 0:
            ax.set_ylabel(f"{dom}\ndensity (n = {int(s.n):,})")
    ax = axes[i, 2]
    keys = [("empirical", "protein"), ("empirical", "three_di"), ("iid", "protein"), ("iid", "three_di")]
    cols = ["#0072B2", "#56B4E9", "#D55E00", "#F0A07A"]
    w = 0.2
    for k, (ref, a) in enumerate(keys):
        g = dec[(dec.domain == dom) & (dec.reference == ref) & (dec.alphabet == a)].sort_values("decile")
        ax.bar(np.arange(10) + (k - 1.5) * w, 100 * g.n / g.n.sum(), width=w, color=cols[k],
               label=f"{'i.i.d.' if ref == 'iid' else ref}, {'3Di' if a == 'three_di' else a}")
    ax.axhline(10, color=INK, lw=0.7, ls="--")
    ax.set_yscale("log")
    ax.set_ylim(0.3, 450)
    ax.set_yticks([0.5, 1, 2, 5, 10, 20, 50, 100], ["0.5", "1", "2", "5", "10", "20", "50", "100"])
    ax.set_xticks(range(10), [str(10 * (k + 1)) for k in range(10)], fontsize=6)
    ax.set_ylabel("% of ORFs in decile")
    if i == 0:
        ax.set_title("percentile deciles (10% = calibrated)")
        ax.legend(frameon=False, fontsize=5.6, ncol=2, loc="upper center")
    else:
        ax.set_xlabel("percentile decile")
panel(axes[0, 0], "a")
panel(axes[1, 0], "b")
fig.tight_layout()
save(fig, "figGS02_reference_calibration")


In [ ]:
print(sm.round(4).to_string(index=False))


## Caption (draft)

**Figure GS02. Only the empirical length-conditioned reference is calibrated.**

Matched ORFs from the systematic figure sample are scored against two references at their own length: the empirical distribution of real matched ORFs, and i.i.d. multinomial draws at the pooled composition. **(a)** Bacteria, n = 1,028,059. **(b)** Archaea, n = 315,638.

- **Left and centre:** the distribution of z against each reference, with N(0, 1) dashed. The empirical z has mean ≈ 0 and SD 1.00 for both protein and 3Di. It is left-skewed, so tail probabilities should be read from the percentile, not from a normal table. The i.i.d. z has an SD of 3.9–6.6: real proteins vary in composition far more than multinomial sampling allows.
- **Right:** the share of ORFs in each percentile decile, where 10% per decile means the reference is calibrated. The empirical reference is flat. The i.i.d. reference piles protein into the lowest decile and splits 3Di and 12-state between the two extremes.
